# Baseline Modeling - Theta & SARIMA

## Configuration:
- **Daily**: SARIMA, Test = 4 weeks, Linear Interpolation
- **Weekly**: THETA, Test = 52 weeks

## Gap Strategy:
- **< 5% gaps**: Linear Interpolation (realistic)
- **>= 5% gaps**: Dropna (too many gaps)

Helper functions live in `src/Favorita_TSA/models/baseline.py`.

In [1]:
# IMPORTS
import os
from pathlib import Path

import mlflow
import pandas as pd
import plotly.graph_objects as go

from Favorita_TSA.models.baseline import (
    aggregate_to_weekly,
    run_baseline_plotly,
)
from Favorita_TSA.models.data_preparation import build_dataframes
from Favorita_TSA.viz.ploty_theme import set_plotly_theme

set_plotly_theme()
print("Imports loaded")

/Users/patrickhederer/Python Projects/Group-Work-Favorita-Forecasting/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports loaded


In [2]:
# CONFIGURATION

ITEMS_TO_MODEL = {
    "daily_smooth": {"store": 25, "item": 115611},
    "daily_erratic": {"store": 44, "item": 103520},
    "weekly_smooth": {"store": 24, "item": 1503844},
    "weekly_erratic": {"store": 51, "item": 1239986},
}

TEST_WEEKS_DAILY = 4
TEST_WEEKS_WEEKLY = 52
GAP_THRESHOLD = 0.05  # 5 %

# Plots werden zusaetzlich als HTML-Datei gespeichert
PROJECT_ROOT = Path("..").resolve()
IMG_DIR = PROJECT_ROOT / "img" / "mlflow"

print("Configuration loaded")
print(f"   Daily:  SARIMA, Test = {TEST_WEEKS_DAILY} weeks")
print(f"   Weekly: THETA,  Test = {TEST_WEEKS_WEEKLY} weeks")
print(
    f"   Gap strategy: Linear (<{GAP_THRESHOLD * 100:.0f}%) | Dropna (>={GAP_THRESHOLD * 100:.0f}%)"
)
print(f"   Plot output: {IMG_DIR}")

Configuration loaded
   Daily:  SARIMA, Test = 4 weeks
   Weekly: THETA,  Test = 52 weeks
   Gap strategy: Linear (<5%) | Dropna (>=5%)
   Plot output: /Users/patrickhederer/Python Projects/Group-Work-Favorita-Forecasting/img/mlflow


In [3]:
# DATEN LADEN

os.chdir(PROJECT_ROOT)

dfs = build_dataframes()

daily_smooth = dfs["smooth_daily"]
daily_erratic = dfs["erratic_daily"]
weekly_smooth = dfs["smooth_weekly"]
weekly_erratic = dfs["erratic_weekly"]

print(f"Daily Smooth:   {daily_smooth.shape}")
print(f"Daily Erratic:  {daily_erratic.shape}")
print(f"Weekly Smooth:  {weekly_smooth.shape}")
print(f"Weekly Erratic: {weekly_erratic.shape}")

Loading fact table …
Loading forecastability matrices …
Building DataFrames …
  smooth_daily         36,560 store-item pairs     39,523,827 rows
  erratic_daily        35,933 store-item pairs     36,705,401 rows
  smooth_weekly        47,196 store-item pairs     30,042,814 rows
  erratic_weekly       19,734 store-item pairs     10,182,367 rows
Daily Smooth:   (39523827, 12)
Daily Erratic:  (36705401, 12)
Weekly Smooth:  (30042814, 12)
Weekly Erratic: (10182367, 12)


In [4]:
# MLFLOW SETUP

MLRUNS_DIR = PROJECT_ROOT / "mlruns"
mlflow.set_tracking_uri(f"file://{MLRUNS_DIR.as_posix()}")
mlflow.set_experiment("favorita_baseline_store_item")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

MLflow tracking URI: file:///Users/patrickhederer/Python Projects/Group-Work-Favorita-Forecasting/mlruns


/Users/patrickhederer/Python Projects/Group-Work-Favorita-Forecasting/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [5]:
# WEEKLY AGGREGATION

print("Weekly Smooth:")
weekly_smooth_agg = aggregate_to_weekly(
    weekly_smooth,
    store=ITEMS_TO_MODEL["weekly_smooth"]["store"],
    item=ITEMS_TO_MODEL["weekly_smooth"]["item"],
)

print("\nWeekly Erratic:")
weekly_erratic_agg = aggregate_to_weekly(
    weekly_erratic,
    store=ITEMS_TO_MODEL["weekly_erratic"]["store"],
    item=ITEMS_TO_MODEL["weekly_erratic"]["item"],
)

print(
    f"\nWeekly Smooth:   {weekly_smooth_agg.shape if weekly_smooth_agg is not None else 'None'}"
)
print(
    f"Weekly Erratic:  {weekly_erratic_agg.shape if weekly_erratic_agg is not None else 'None'}"
)

Weekly Smooth:
   Input frequency: 1 day(s) between observations
   Aggregated: 993 daily obs -> 147 weekly obs (2013-12-30 to 2017-08-14)

Weekly Erratic:
   Input frequency: 7 day(s) between observations
   Aggregated: 184 daily obs -> 179 weekly obs (2013-11-04 to 2017-07-31)

Weekly Smooth:   (147, 4)
Weekly Erratic:  (179, 4)


## Run Daily Smooth

In [6]:
# Filter out the post-earthquake data gap present in this store-item
daily_smooth_clean = daily_smooth[daily_smooth["date"] < "2016-08-22"].copy()

In [7]:
result_daily_smooth, fig_daily_smooth = run_baseline_plotly(
    df=daily_smooth_clean,
    pattern="daily_smooth",
    store=ITEMS_TO_MODEL["daily_smooth"]["store"],
    item=ITEMS_TO_MODEL["daily_smooth"]["item"],
    freq="D",
    season_length=7,
    test_weeks=TEST_WEEKS_DAILY,
    model_type="sarima",
    gap_threshold=GAP_THRESHOLD,
    img_dir=IMG_DIR,
)
fig_daily_smooth.show()


DAILY_SMOOTH | SARIMA
   Deleted FAILED run: 1acffe63...
Run name: daily_smooth_002_sarima
MLflow run_id: 184e609c175f42288a0932fad04f2ec5
   Using column: 'date' (freq=D)
Loaded: 1284 obs | 2013-01-01 to 2016-08-21
   Mean: 10.02, Std: 6.27
   Filled 45 daily gaps with linear interpolation
   Final: 1329 observations
Split: 1301 train | 28 test
   Train: 2013-01-01 to 2016-07-24
   Test:  2016-07-25 to 2016-08-21

Training AutoARIMA (season_length=7, {}) ...
   Forecasting 28 periods ...
   Primary -- MAE: 4.01 | R2: -0.067

Training SeasonalNaive ...
   Naive  -- MAE: 4.61 | R2: -0.597

Improvement vs Naive: +12.9%
   Plot saved: /Users/patrickhederer/Python Projects/Group-Work-Favorita-Forecasting/img/mlflow/daily_smooth_002_sarima.html
   MLflow run finished: daily_smooth_002_sarima


## Run Daily Erratic

In [8]:
result_daily_erratic, fig_daily_erratic = run_baseline_plotly(
    df=daily_erratic,
    pattern="daily_erratic",
    store=ITEMS_TO_MODEL["daily_erratic"]["store"],
    item=ITEMS_TO_MODEL["daily_erratic"]["item"],
    freq="D",
    season_length=7,
    test_weeks=TEST_WEEKS_DAILY,
    model_type="sarima",
    gap_threshold=GAP_THRESHOLD,
    img_dir=IMG_DIR,
)
fig_daily_erratic.show()


DAILY_ERRATIC | SARIMA
Run name: daily_erratic_002_sarima
MLflow run_id: 478d42cbfebb4f1e804e27cb33abbb4a
   Using column: 'date' (freq=D)
Loaded: 1580 obs | 2013-01-02 to 2017-08-15
   Mean: 8.73, Std: 7.26
   107 gaps (6.3%) exceed threshold (5%) -- dropping NaN rows
   Final: 1580 observations
Split: 1552 train | 28 test
   Train: 2013-01-02 to 2017-07-18
   Test:  2017-07-19 to 2017-08-15

Training AutoARIMA (season_length=7, {}) ...
   Forecasting 28 periods ...
   Primary -- MAE: 2.58 | R2: -0.195

Training SeasonalNaive ...
   Naive  -- MAE: 2.82 | R2: -0.401

Improvement vs Naive: +8.6%
   Plot saved: /Users/patrickhederer/Python Projects/Group-Work-Favorita-Forecasting/img/mlflow/daily_erratic_002_sarima.html
   MLflow run finished: daily_erratic_002_sarima


## Run Weekly Smooth (THETA)

In [9]:
result_weekly_smooth, fig_weekly_smooth = run_baseline_plotly(
    df=weekly_smooth_agg,
    pattern="weekly_smooth",
    store=ITEMS_TO_MODEL["weekly_smooth"]["store"],
    item=ITEMS_TO_MODEL["weekly_smooth"]["item"],
    freq="W",
    season_length=52,
    test_weeks=TEST_WEEKS_WEEKLY,
    model_type="theta",
    gap_threshold=GAP_THRESHOLD,
    img_dir=IMG_DIR,
)
fig_weekly_smooth.show()


WEEKLY_SMOOTH | THETA
Run name: weekly_smooth_002_theta
MLflow run_id: 8cccee55c8c543748f3378e96d762111
   Using column: 'week_start' (freq=W)
Loaded: 147 obs | 2013-12-30 to 2017-08-14
   Mean: 1680.93, Std: 299.56
   Weekly data ready (no gap filling needed)
   Final: 147 observations
Split: 95 train | 52 test
   Train: 2013-12-30 to 2016-08-15
   Test:  2016-08-22 to 2017-08-14

Training Theta (season_length=52, {}) ...
   Forecasting 52 periods ...
   Primary -- MAE: 161.74 | R2: -0.114

Training SeasonalNaive ...
   Naive  -- MAE: 191.33 | R2: -0.552

Improvement vs Naive: +15.5%
   Plot saved: /Users/patrickhederer/Python Projects/Group-Work-Favorita-Forecasting/img/mlflow/weekly_smooth_002_theta.html
   MLflow run finished: weekly_smooth_002_theta


## Run Weekly Erratic (THETA)

In [10]:
result_weekly_erratic, fig_weekly_erratic = run_baseline_plotly(
    df=weekly_erratic_agg,
    pattern="weekly_erratic",
    store=ITEMS_TO_MODEL["weekly_erratic"]["store"],
    item=ITEMS_TO_MODEL["weekly_erratic"]["item"],
    freq="W",
    season_length=52,
    test_weeks=TEST_WEEKS_WEEKLY,
    model_type="theta",
    gap_threshold=GAP_THRESHOLD,
    img_dir=IMG_DIR,
)
fig_weekly_erratic.show()


WEEKLY_ERRATIC | THETA
Run name: weekly_erratic_002_theta
MLflow run_id: b84ed8a10c6d45b48489bbbfc7b4a7f6
   Using column: 'week_start' (freq=W)
Loaded: 179 obs | 2013-11-04 to 2017-07-31
   Mean: 1223.06, Std: 1419.62
   Weekly data ready (no gap filling needed)
   Final: 179 observations
Split: 132 train | 47 test
   Train: 2013-11-04 to 2016-08-01
   Test:  2016-08-08 to 2017-07-31

Training Theta (season_length=52, {}) ...
   Forecasting 47 periods ...
   Primary -- MAE: 546.72 | R2: -0.171

Training SeasonalNaive ...
   Naive  -- MAE: 1948.37 | R2: -7.267

Improvement vs Naive: +71.9%
   Plot saved: /Users/patrickhederer/Python Projects/Group-Work-Favorita-Forecasting/img/mlflow/weekly_erratic_002_theta.html
   MLflow run finished: weekly_erratic_002_theta


## Summary

In [11]:
results = [
    result_daily_smooth,
    result_daily_erratic,
    result_weekly_smooth,
    result_weekly_erratic,
]

summary_df = pd.DataFrame(results)

print("=" * 70)
print("BASELINE SUMMARY")
print("=" * 70)
print(
    summary_df[
        [
            "pattern",
            "model_type",
            "test_weeks",
            "test_size",
            "mae_primary",
            "mae_naive",
            "improvement_pct",
        ]
    ].to_string(index=False)
)

bar_colors = ["#e74c3c" if "daily" in p else "#9b59b6" for p in summary_df["pattern"]]

fig_summary = go.Figure()
fig_summary.add_trace(
    go.Bar(
        x=summary_df["pattern"],
        y=summary_df["mae_primary"],
        name="Primary Model",
        marker_color=bar_colors,
        opacity=0.85,
        text=summary_df["mae_primary"].round(2),
        textposition="outside",
        customdata=summary_df["model_type"],
        hovertemplate="%{x}<br>%{customdata} MAE: %{y:.2f}<extra></extra>",
    )
)
fig_summary.add_trace(
    go.Bar(
        x=summary_df["pattern"],
        y=summary_df["mae_naive"],
        name="Naive",
        marker_color="#f39c12",
        opacity=0.6,
        text=summary_df["mae_naive"].round(2),
        textposition="outside",
    )
)

for i, row in summary_df.iterrows():
    fig_summary.add_annotation(
        x=i,
        y=max(row["mae_primary"], row["mae_naive"]) * 1.05,
        text=f"{row['improvement_pct']:+.1f}%",
        showarrow=False,
        font={
            "size": 14,
            "color": "#2ecc71" if row["improvement_pct"] > 0 else "#e74c3c",
            "family": "Arial Black",
        },
    )

fig_summary.update_layout(
    title="Baseline Model Comparison<br><sub>Daily: SARIMA | Weekly: THETA</sub>",
    xaxis_title="Pattern",
    yaxis_title="MAE",
    barmode="group",
    height=500,
)
fig_summary.show()

summary_df.to_csv("data/baseline_results/baseline_summary_final.csv", index=False)
print("Saved: data/baseline_results/baseline_summary_final.csv")

BASELINE SUMMARY
       pattern model_type  test_weeks  test_size  mae_primary   mae_naive  improvement_pct
  daily_smooth     sarima           4         28     4.014931    4.607143        12.854208
 daily_erratic     sarima           4         28     2.580090    2.821429         8.553784
 weekly_smooth      theta          52         52   161.736348  191.326038        15.465584
weekly_erratic      theta          52         47   546.720952 1948.370404        71.939578


Saved: data/baseline_results/baseline_summary_final.csv


In [12]:
summary_df

,pattern,model_type,freq,store,item,season_length,test_weeks,train_size,test_size,mae_naive,r2_naive,mae_primary,r2_primary,improvement_pct
0,daily_smooth,sarima,D,25,115611,7,4,1301,28,4.607143,-0.596533,4.014931,-0.067358,12.854208
1,daily_erratic,sarima,D,44,103520,7,4,1552,28,2.821429,-0.401477,2.580090,-0.194544,8.553784
2,weekly_smooth,theta,W,24,1503844,52,52,95,52,191.326038,-0.552299,161.736348,-0.114038,15.465584
3,weekly_erratic,theta,W,51,1239986,52,52,132,47,1948.370404,-7.267047,546.720952,-0.171211,71.939578
